In [1]:
import pandas

/Users/karthickkumarasamy/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv('Detailed_Polling_Data.csv')

<IPython.core.display.Javascript object>

In [3]:
df.columns

Index(['Serial No. Of Polling Station', 'Dravida Munnetra Kazhagam',
       'All India Anna Dravida Munnetra Kazhagam', 'Naam Tamilar Katchi',
       'All India Puratchi Thalaivar Makkal Munnettra Kazhagam',
       'Tamizhaga Vaazhvurimai Katchi',
       'Communist Party of India (Marxist-Leninist) (Liberation)',
       'Makkal Nalvaazhvuk Katchi',
       'Nam Naadu Nam Makkal Nam Ethirkaalam Katchi', 'Naam Indiar Party',
       'Tamilaga Vettri Kazhagam', 'Independent', 'Independent.1',
       'Independent.2', 'Independent.3', 'Independent.4',
       'Total of Valid Votes', 'No. Of Rejected Votes', 'NOTA', 'Total',
       'No. Of Tendered Votes', 'Locality',
       'Building in Which it will be Located', 'Polling Area',
       'Whether for all Voters or men olny or women only',
       'Dravida Munnetra Kazhagam_Share_%',
       'All India Anna Dravida Munnetra Kazhagam_Share_%',
       'Naam Tamilar Katchi_Share_%',
       'All India Puratchi Thalaivar Makkal Munnettra Kazhagam_Share_

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Load the ninth dataset (Update the filename to match your file)
df9 = pd.read_csv("Detailed_Polling_Data.csv")

# 2. Select the core primary political party share columns present in this index
new_party_shares = [
    'Dravida Munnetra Kazhagam_Share_%',
    'All India Anna Dravida Munnetra Kazhagam_Share_%',
    'Tamilaga Vettri Kazhagam_Share_%',
    'Naam Tamilar Katchi_Share_%'
]

# Handle any missing data in the share columns by making them 0
df9[new_party_shares] = df9[new_party_shares].fillna(0)

# 3. Include structural features for the voter behavior analysis
# We utilize your pre-calculated Independent_Share_% and Margin_Percentage columns
feature_cols = new_party_shares + ['Independent_Share_%', 'Margin_Percentage']
df9[feature_cols] = df9[feature_cols].fillna(0)

# 4. Extract and scale the features
X = df9[feature_cols]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5. Apply K-Means Clustering to group booths into 4 core segments
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df9['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# 6. Print the Raw Profile Breakdown to help map the text identities
print("\n--- DATASET 9: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---")
profile = df9.groupby('Cluster_ID')[feature_cols].mean()
print(profile.round(2))

print("\n--- DATASET 9: BOOTH COUNT PER CLUSTER ---")
print(df9['Cluster_ID'].value_counts())

# 7. Automatically export separate action lists for campaign ground teams
for cluster_num in range(optimal_k):
    cluster_df = df9[df9['Cluster_ID'] == cluster_num][
        [
            'Serial No. Of Polling Station', 
            'Locality', 
            'Building in Which it will be Located', 
            'Polling Area', 
            'Winner_Party', 
            'Margin_Percentage'
        ]
    ]
    filename = f"Dataset_9_Cluster_{cluster_num}_Booths.csv"
    cluster_df.to_csv(filename, index=False)

print("\nSuccess! Campaign target files generated for all 4 clusters.")
